# Silent Signal — chuẩn bị và kiểm tra VSL400 trên Google Colab

Notebook này chạy pipeline đã triển khai trong dự án: tạo manifest, tạo label map, kiểm tra cấu trúc ba view, kiểm tra file và chia signer-disjoint. Dữ liệu video được đọc từ Google Drive hoặc một thư mục cloud đã mount; không cần tải xuống laptop.

Trước khi chạy, bạn cần được cấp quyền VSL400 và tuân thủ Data Usage Agreement tại https://zenodo.org/records/21366957. Notebook giả định dataset đã được giải nén thành front_view, left_view, right_view và ba file JSON tương ứng. Không ghi token truy cập vào cell.

Nếu bạn nhận các file Part_*.zip, hãy giải nén/ghép đúng theo README và merge_splits.py do tác giả cung cấp. Không giả định rằng từng part là một ZIP độc lập. Mỗi view có khoảng 24.753 file, nên đọc trực tiếp nhiều file nhỏ từ Drive có thể chậm hoặc gặp giới hạn I/O. Khi ổ /content đủ lớn, có thể đặt DATASET_ROOT tại /content/VSL400 và vẫn lưu toàn bộ kết quả về Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Nơi lấy source code: drive hoặc git.
PROJECT_SOURCE = 'drive'  # @param ['drive', 'git']
PROJECT_DRIVE_DIR = Path('/content/drive/MyDrive/silent-signal')
PROJECT_GIT_URL = ''  # @param {type:'string'}
PROJECT_GIT_REF = 'main'  # @param {type:'string'}

# Dataset phải là thư mục đã giải nén. Dùng /content/VSL400 nếu tải trực tiếp.
DATASET_ROOT = Path('/content/VSL400')
RESULT_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl400')

# Tải trực tiếp từ Zenodo sau khi tài khoản đã được duyệt quyền.
DOWNLOAD_FROM_ZENODO = False  # @param {type:'boolean'}
DOWNLOAD_LARGE_PARTS = False  # @param {type:'boolean'}
VERIFY_ZENODO_MD5 = True  # @param {type:'boolean'}
ZENODO_RECORD_ID = '17943574'
ZENODO_DOWNLOAD_DIR = Path('/content/vsl400-release')

# Để trống nếu chưa có split chính thức từ tác giả.
OFFICIAL_SPLIT_PATH = ''  # @param {type:'string'}

PROBE_WORKERS = 2  # @param {type:'integer'}
PROBE_BATCH_SIZE = 250  # @param {type:'integer'}

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_ROOT)
print('Kết quả bền vững:', RESULT_ROOT)

## Cài source code và công cụ

Nếu chọn drive, hãy đưa thư mục dự án silent-signal lên MyDrive. Nếu chọn git, điền URL repository. Source được chép/clone vào ổ tạm của Colab để cài nhanh hơn.

In [ ]:
import shutil
import subprocess
import sys

PROJECT_ROOT = Path('/content/silent-signal')

if PROJECT_SOURCE == 'drive':
    if not (PROJECT_DRIVE_DIR / 'pyproject.toml').is_file():
        raise FileNotFoundError(
            f'Không tìm thấy project tại {PROJECT_DRIVE_DIR}. '
            'Hãy đưa source code lên Drive hoặc chọn PROJECT_SOURCE=git.'
        )
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        PROJECT_DRIVE_DIR,
        PROJECT_ROOT,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('.venv', '.git', '__pycache__', 'artifacts'),
    )
elif PROJECT_SOURCE == 'git':
    if not PROJECT_GIT_URL:
        raise ValueError('Hãy điền PROJECT_GIT_URL.')
    if not (PROJECT_ROOT / '.git').is_dir():
        subprocess.run(
            ['git', 'clone', '--branch', PROJECT_GIT_REF, '--depth', '1',
             PROJECT_GIT_URL, str(PROJECT_ROOT)],
            check=True,
        )
else:
    raise ValueError("PROJECT_SOURCE phải là 'drive' hoặc 'git'.")

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(PROJECT_ROOT)],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

if shutil.which('ffprobe') is None or shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

print('Project:', PROJECT_ROOT)
print('ffprobe:', shutil.which('ffprobe'))
print('ffmpeg:', shutil.which('ffmpeg'))

## Tải VSL400 trực tiếp từ Zenodo (tùy chọn)

VSL400 là record restricted. Trước tiên hãy đăng nhập Zenodo, gửi yêu cầu truy cập và chờ chủ dữ liệu duyệt. Sau đó tạo Personal Access Token bằng chính tài khoản đã được duyệt. Trong Colab, mở biểu tượng chìa khóa **Secrets**, tạo secret tên `ZENODO_TOKEN`, dán token và bật Notebook access. Không ghi token trực tiếp vào notebook hoặc Git.

Đặt `DOWNLOAD_FROM_ZENODO=True`. Lần đầu nên giữ `DOWNLOAD_LARGE_PARTS=False` để xác nhận quyền và tải README, script ghép cùng source baseline nhỏ. Khi thành công, đổi thành `True` để tải bảy part lớn. Downloader hỗ trợ HTTP Range để tiếp tục file dở trong cùng runtime và kiểm tra MD5. Dữ liệu trong `/content` sẽ mất khi runtime bị xóa; tổng archive khoảng 70,8 GB và quá trình giải nén cần thêm dung lượng.

Sau khi tải, đọc `README.txt` được in ở cuối cell và chạy `merge_splits.py` đúng theo hướng dẫn chính thức. Không tự giải nén riêng từng part khi README không yêu cầu. Cuối cùng bảo đảm `DATASET_ROOT` trỏ tới thư mục chứa `front_view`, `left_view`, `right_view` và ba JSON metadata.

In [ ]:
if DOWNLOAD_FROM_ZENODO:
    import hashlib
    import time
    from urllib.parse import quote

    import requests
    from google.colab import userdata
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    try:
        zenodo_token = userdata.get('ZENODO_TOKEN')
    except Exception as exc:
        raise RuntimeError(
            'Thiếu Colab Secret ZENODO_TOKEN. Tạo secret rồi bật Notebook access.'
        ) from exc
    if not zenodo_token:
        raise RuntimeError('ZENODO_TOKEN đang rỗng.')

    session = requests.Session()
    session.headers.update({
        'Authorization': f'Bearer {zenodo_token}',
        'User-Agent': 'silent-signal-colab/0.1',
    })
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=2,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({'GET'}),
    )
    session.mount('https://', HTTPAdapter(max_retries=retry))

    def zenodo_entries(payload):
        files = payload.get('files', payload) if isinstance(payload, dict) else payload
        if isinstance(files, list):
            return files
        if not isinstance(files, dict):
            return []
        entries = files.get('entries', [])
        if isinstance(entries, dict):
            return list(entries.values())
        return entries if isinstance(entries, list) else []

    api_root = f'https://zenodo.org/api/records/{ZENODO_RECORD_ID}'
    response = session.get(api_root, timeout=(30, 120))
    if response.status_code in (401, 403):
        raise PermissionError(
            'Zenodo từ chối quyền. Hãy kiểm tra token và việc chủ VSL400 đã duyệt tài khoản.'
        )
    response.raise_for_status()
    entries = zenodo_entries(response.json())
    if not entries:
        response = session.get(f'{api_root}/files', timeout=(30, 120))
        if response.status_code in (401, 403):
            raise PermissionError('Tài khoản chưa có quyền đọc file VSL400.')
        response.raise_for_status()
        entries = zenodo_entries(response.json())
    if not entries:
        raise RuntimeError(
            'API không trả về file. Record vẫn restricted đối với tài khoản/token này.'
        )

    expected_large = {f'Part_{index}.zip' for index in range(1, 8)}
    expected_small = {
        'README.txt',
        'merge_splits.py',
        'VietnameseSignLanguageRecognition.zip',
    }
    selected_names = expected_small | (expected_large if DOWNLOAD_LARGE_PARTS else set())
    entries_by_name = {
        str(item.get('key') or item.get('filename')): item
        for item in entries
        if item.get('key') or item.get('filename')
    }
    missing_names = sorted(selected_names - set(entries_by_name))
    if missing_names:
        raise RuntimeError(f'Zenodo không trả về các file cần thiết: {missing_names}')

    ZENODO_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    selected = [entries_by_name[name] for name in sorted(selected_names)]
    remaining_bytes = sum(
        max(int(item.get('size') or 0) - (ZENODO_DOWNLOAD_DIR / str(item.get('key') or item.get('filename'))).stat().st_size, 0)
        if (ZENODO_DOWNLOAD_DIR / str(item.get('key') or item.get('filename'))).is_file()
        else int(item.get('size') or 0)
        for item in selected
    )
    free_bytes = shutil.disk_usage(ZENODO_DOWNLOAD_DIR).free
    print(f'Cần tải thêm: {remaining_bytes / 1024**3:.2f} GiB')
    print(f'Dung lượng trống tại đích: {free_bytes / 1024**3:.2f} GiB')
    if remaining_bytes and free_bytes < remaining_bytes * 1.03:
        raise RuntimeError(
            'Không đủ dung lượng cho archive. Dùng runtime disk lớn hơn hoặc đổi '
            'ZENODO_DOWNLOAD_DIR sang storage có đủ chỗ.'
        )

    def verify_md5(path, checksum):
        if not checksum or not str(checksum).startswith('md5:'):
            return
        expected = str(checksum).split(':', 1)[1].lower()
        digest = hashlib.md5()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(16 * 1024 * 1024), b''):
                digest.update(chunk)
        if digest.hexdigest().lower() != expected:
            raise RuntimeError(f'MD5 không khớp: {path.name}')

    def download_entry(item):
        name = str(item.get('key') or item.get('filename'))
        destination = ZENODO_DOWNLOAD_DIR / name
        expected_size = int(item.get('size') or 0)
        current_size = destination.stat().st_size if destination.is_file() else 0
        if expected_size and current_size > expected_size:
            raise RuntimeError(f'File local lớn hơn metadata Zenodo: {destination}')
        content_url = item.get('links', {}).get('content')
        if not content_url:
            content_url = f'{api_root}/files/{quote(name, safe="")}/content'

        if not expected_size or current_size != expected_size:
            headers = {'Range': f'bytes={current_size}-'} if current_size else {}
            with session.get(
                content_url, headers=headers, stream=True, timeout=(30, 300)
            ) as stream:
                if stream.status_code == 416 and expected_size == current_size:
                    pass
                else:
                    stream.raise_for_status()
                    append = current_size > 0 and stream.status_code == 206
                    if current_size > 0 and not append:
                        print(f'{name}: server không nhận Range, tải lại từ đầu')
                        current_size = 0
                    mode = 'ab' if append else 'wb'
                    downloaded = current_size
                    next_report = downloaded + 512 * 1024**2
                    with destination.open(mode) as handle:
                        for chunk in stream.iter_content(chunk_size=16 * 1024 * 1024):
                            if not chunk:
                                continue
                            handle.write(chunk)
                            downloaded += len(chunk)
                            if downloaded >= next_report:
                                total = f'/{expected_size / 1024**3:.2f}' if expected_size else ''
                                print(f'{name}: {downloaded / 1024**3:.2f}{total} GiB')
                                next_report += 512 * 1024**2

        final_size = destination.stat().st_size
        if expected_size and final_size != expected_size:
            raise RuntimeError(
                f'Size không khớp cho {name}: {final_size} != {expected_size}'
            )
        if VERIFY_ZENODO_MD5:
            print(f'{name}: kiểm tra MD5...')
            verify_md5(destination, item.get('checksum'))
        print(f'{name}: hoàn tất')

    for entry in selected:
        download_entry(entry)

    readme_path = ZENODO_DOWNLOAD_DIR / 'README.txt'
    if readme_path.is_file():
        print('\n===== README chính thức của VSL400 =====\n')
        print(readme_path.read_text(encoding='utf-8', errors='replace'))
    print('File đã tải tại:', ZENODO_DOWNLOAD_DIR)
else:
    print('DOWNLOAD_FROM_ZENODO=False: bỏ qua tải dataset.')

## Kiểm tra vị trí dataset

Cell này chỉ xác nhận cấu trúc đầu vào. Pipeline vẫn kiểm tra đầy đủ số video, signer, gloss và quan hệ ba view ở bước kế tiếp.

In [ ]:
import os

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Không tìm thấy DATASET_ROOT: {DATASET_ROOT}')

layout_candidates = {
    'front': (('front_view', 'front_view.json'), ('cam_1', 'cam_1.json')),
    'left': (('left_view', 'left_view.json'), ('cam_2', 'cam_2.json')),
    'right': (('right_view', 'right_view.json'), ('cam_3', 'cam_3.json')),
}

resolved_layout = {}
for view, candidates in layout_candidates.items():
    for directory_name, metadata_name in candidates:
        if ((DATASET_ROOT / directory_name).is_dir()
                and (DATASET_ROOT / metadata_name).is_file()):
            resolved_layout[view] = (directory_name, metadata_name)
            break
    if view not in resolved_layout:
        raise FileNotFoundError(
            f'Thiếu directory hoặc metadata cho view {view}: {candidates}'
        )

runtime_disk = shutil.disk_usage('/content')
print('Layout:', resolved_layout)
print(f'Ổ tạm Colab còn: {runtime_disk.free / 1024**3:.1f} GiB')
print('VSL400_ROOT:', os.fspath(DATASET_ROOT))

## Bước 1 — tạo manifest, kiểm tra metadata và chia signer

Lệnh all tạo CSV/Parquet, label map, báo cáo lỗi và split. Nếu phát hiện lỗi, pipeline dừng trước khi tạo split. Các artifact vẫn được sao chép về Drive để bạn xem báo cáo.

In [ ]:
import json

CONFIG_PATH = PROJECT_ROOT / 'configs/dataset/vsl400.yaml'
command = [
    sys.executable,
    '-m',
    'silent_signal.cli.prepare',
    'all',
    '--config',
    str(CONFIG_PATH),
    '--root',
    str(DATASET_ROOT),
    '--level',
    'metadata',
]
if OFFICIAL_SPLIT_PATH:
    official_path = Path(OFFICIAL_SPLIT_PATH)
    if not official_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy official split: {official_path}')
    command.extend(['--official-split', str(official_path)])

completed = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)

artifact_map = {
    'data/manifests/vsl400.csv': 'manifests/vsl400.csv',
    'data/manifests/vsl400.parquet': 'manifests/vsl400.parquet',
    'data/labels/vsl400_labels.json': 'labels/vsl400_labels.json',
    'data/splits/vsl400_signer_split.json': 'splits/vsl400_signer_split.json',
    'artifacts/runs/data-preparation/validation_report.json':
        'reports/metadata_validation_report.json',
    'artifacts/runs/data-preparation/invalid_records.csv':
        'reports/metadata_invalid_records.csv',
}
for source_name, destination_name in artifact_map.items():
    source = PROJECT_ROOT / source_name
    if source.is_file():
        destination = RESULT_ROOT / destination_name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

if completed.returncode != 0:
    raise RuntimeError(
        'Kiểm tra metadata không đạt. Xem reports/metadata_validation_report.json '
        'và reports/metadata_invalid_records.csv trong RESULT_ROOT.'
    )

print('Đã hoàn thành metadata + signer split.')
print('Kết quả:', RESULT_ROOT)

In [ ]:
import pandas as pd

manifest_path = RESULT_ROOT / 'manifests/vsl400.parquet'
manifest_frame = pd.read_parquet(manifest_path)
display(manifest_frame.head(6))
display(
    manifest_frame.groupby('split', dropna=False).agg(
        clips=('sample_id', 'count'),
        instances=('instance_id', 'nunique'),
        signers=('signer_id', 'nunique'),
        glosses=('class_index', 'nunique'),
    )
)

report_path = RESULT_ROOT / 'reports/metadata_validation_report.json'
metadata_report = json.loads(report_path.read_text(encoding='utf-8'))
print('Passed:', metadata_report['passed'])
print('Issue counts:', metadata_report['issue_counts'])

## Bước 2 — ffprobe có thể tiếp tục sau khi Colab ngắt

Đổi RUN_PROBE thành True để đọc header toàn bộ video. Mỗi batch được ghi ngay vào probe_cache.jsonl trên Drive. Khi chạy lại notebook, các sample đã có cache sẽ được bỏ qua. Với dataset nằm trên Drive, nên bắt đầu bằng 1–2 worker để tránh giới hạn I/O.

In [ ]:
RUN_PROBE = False  # @param {type:'boolean'}

from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict

from silent_signal.data.manifest import read_manifest
from silent_signal.data.validation import probe_video

probe_cache_path = RESULT_ROOT / 'reports/probe_cache.jsonl'
records = read_manifest(RESULT_ROOT / 'manifests/vsl400.parquet')

def load_jsonl_cache(path):
    cache = {}
    if not path.is_file():
        return cache
    with path.open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                print(f'Bỏ qua dòng cache lỗi: {line_number}')
                continue
            cache[item['sample_id']] = item
    return cache

def video_signature(record):
    video_path = DATASET_ROOT / record.video_path
    if not video_path.is_file():
        return {'file_size_bytes': None, 'mtime_ns': None}
    stat = video_path.stat()
    return {'file_size_bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}

def cache_is_current(record, item):
    signature = video_signature(record)
    return all(item.get(key) == value for key, value in signature.items())

def probe_record(record):
    signature = video_signature(record)
    try:
        media = probe_video(DATASET_ROOT / record.video_path)
        return {
            'sample_id': record.sample_id,
            'status': 'ok',
            'media': asdict(media),
            **signature,
        }
    except Exception as exc:
        return {
            'sample_id': record.sample_id,
            'status': 'error',
            'error_type': type(exc).__name__,
            'error': str(exc)[:2000],
            **signature,
        }

probe_cache = load_jsonl_cache(probe_cache_path)
pending = [
    record for record in records
    if (record.sample_id not in probe_cache
        or not cache_is_current(record, probe_cache[record.sample_id]))
]
print(f'Đã cache: {len(probe_cache):,}; còn lại: {len(pending):,}')

if RUN_PROBE:
    probe_cache_path.parent.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(pending), PROBE_BATCH_SIZE):
        batch = pending[start:start + PROBE_BATCH_SIZE]
        batch_results = []
        with ThreadPoolExecutor(max_workers=PROBE_WORKERS) as executor:
            futures = [executor.submit(probe_record, record) for record in batch]
            for future in as_completed(futures):
                batch_results.append(future.result())
        with probe_cache_path.open('a', encoding='utf-8') as handle:
            for item in batch_results:
                handle.write(json.dumps(item, ensure_ascii=False) + '\n')
                probe_cache[item['sample_id']] = item
        checked = min(start + len(batch), len(pending))
        errors = sum(item['status'] == 'error' for item in probe_cache.values())
        print(f'Phiên này: {checked:,}/{len(pending):,}; tổng lỗi probe: {errors:,}')
else:
    print('RUN_PROBE=False: chưa chạy ffprobe. Đổi thành True khi sẵn sàng.')

## Hoàn thiện báo cáo probe

Chỉ bật FINALIZE_PROBE sau khi cache đủ tất cả sample. Cell này đưa kết quả probe qua toàn bộ quy tắc kiểm tra FPS, kích thước, số frame và thời lượng, rồi ghi manifest/report cuối về Drive.

In [ ]:
FINALIZE_PROBE = False  # @param {type:'boolean'}

from silent_signal.configuration import load_dataset_config
from silent_signal.data.manifest import write_manifest
from silent_signal.data.validation import (
    MediaInfo,
    MediaProbeError,
    validate_manifest,
    write_validation_report,
)

if FINALIZE_PROBE:
    probe_cache = load_jsonl_cache(probe_cache_path)
    missing_ids = [
        record.sample_id for record in records
        if (record.sample_id not in probe_cache
            or not cache_is_current(record, probe_cache[record.sample_id]))
    ]
    if missing_ids:
        raise RuntimeError(
            f'Cache còn thiếu {len(missing_ids):,} sample. Hãy chạy RUN_PROBE trước.'
        )

    sample_by_path = {
        str((DATASET_ROOT / record.video_path).resolve()): record.sample_id
        for record in records
    }

    def cached_probe(path):
        sample_id = sample_by_path[str(path.resolve())]
        item = probe_cache[sample_id]
        if item['status'] != 'ok':
            raise MediaProbeError(item.get('error', 'probe failed'))
        return MediaInfo(**item['media'])

    dataset_config = load_dataset_config(CONFIG_PATH, root_override=DATASET_ROOT)
    probe_result = validate_manifest(
        records,
        dataset_root=DATASET_ROOT,
        expected=dataset_config.expected,
        expected_views=tuple(dataset_config.views),
        level='probe',
        workers=8,
        probe_function=cached_probe,
    )

    write_manifest(probe_result.records, RESULT_ROOT / 'manifests/vsl400.probed.csv')
    write_manifest(probe_result.records, RESULT_ROOT / 'manifests/vsl400.probed.parquet')
    write_manifest(
        tuple(record for record in probe_result.records if not record.is_valid),
        RESULT_ROOT / 'reports/probe_invalid_records.csv',
    )
    write_validation_report(
        probe_result,
        RESULT_ROOT / 'reports/probe_validation_report.json',
    )
    print(json.dumps(probe_result.to_report()['summary'], indent=2))
    print('Probe passed:', not probe_result.has_errors)
else:
    print('FINALIZE_PROBE=False: chưa tổng hợp báo cáo probe.')

## Full decode tùy chọn

Full decode 74.259 video có thể vượt thời lượng một phiên Colab. Chỉ bật sau khi probe đã hoàn tất và khi có runtime phù hợp. Lệnh bên dưới chạy lại probe rồi giải mã toàn bộ video; báo cáo sẽ được tạo trong project tạm, vì vậy cần sao chép artifact về RESULT_ROOT sau khi hoàn thành.

In [ ]:
RUN_FULL_DECODE = False  # @param {type:'boolean'}

if RUN_FULL_DECODE:
    decode_command = [
        sys.executable,
        '-m',
        'silent_signal.cli.prepare',
        'validate',
        '--config', str(CONFIG_PATH),
        '--root', str(DATASET_ROOT),
        '--manifest', str(RESULT_ROOT / 'manifests/vsl400.parquet'),
        '--level', 'decode',
        '--workers', '2',
    ]
    decode_completed = subprocess.run(
        decode_command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    print(decode_completed.stdout)
    if decode_completed.stderr:
        print(decode_completed.stderr, file=sys.stderr)

    decode_artifacts = {
        'data/manifests/vsl400.csv': 'manifests/vsl400.decoded.csv',
        'data/manifests/vsl400.parquet': 'manifests/vsl400.decoded.parquet',
        'artifacts/runs/data-preparation/validation_report.json':
            'reports/decode_validation_report.json',
        'artifacts/runs/data-preparation/invalid_records.csv':
            'reports/decode_invalid_records.csv',
    }
    for source_name, destination_name in decode_artifacts.items():
        source = PROJECT_ROOT / source_name
        if source.is_file():
            destination = RESULT_ROOT / destination_name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)

    if decode_completed.returncode != 0:
        raise RuntimeError(
            'Full decode phát hiện lỗi. Xem decode_validation_report.json trên Drive.'
        )
    print('Đã hoàn thành full decode.')
else:
    print('RUN_FULL_DECODE=False: bỏ qua full decode.')